# Set af spaCy til kinesisk tekst

## Script Summary

Dette notebook giver en komplet guide til at installere og bruge spaCy's kinesiske NLP pipeline til at analyse kinesisk tekst. Notebooket dækker:

1. **Installation og Setup**: Installation af spaCy og download af kinesiske modeller
2. **Grundlæggende NLP Funktioner**: Word segmentation, POS tagging, Named Entity Recognition og Dependency Parsing
3. **Avancerede Funktioner**: Batch processing (at behandle flere sætninger på én gang), tokenanalyse og visualisering
4. **Navneord Modifier Analyse**: Funktioner til at finde og analysere ord der modificerer navneord i kinesisk tekst

### Tilgængelige Modeller

SpaCy tilbyder flere kinesiske modeller:
- **zh_core_web_sm** - Lille model (hurtig, mindre hukommelse)
- **zh_core_web_md** - Medium model (med ord embeddings)
- **zh_core_web_lg** - Stor model (store ord embeddings)
- **zh_core_web_trf** - Transformer model (bedste performance, kræver transformers bibliotek)

### Funktioner

Alle modellerne kan håndtere:
- **Ordsegmentering** (word segmentation/tokenization)
- **Ordklasse-tagging** (Part-of-Speech Tagging)
- **Navngiven Entitetsgenkendelse** (Named Entity Recognition)
- **Afhængigheds-parsing** (Dependency Parsing)
- **Syntaktisk analyse** (Syntactic analysis)
- **Navneord modifier analyse** (Find ord der modificerer navneord)

## Installation

Først skal vi installere spaCy og den kinesiske model.

Nedenfor tjekker programmet om spaCy allerede er installeret og installerer det, hvis det ikke er  installeret.

In [1]:
# Installerer spaCy biblioteket
# Cellen køres kun første gang
import sys
import subprocess

def install_package(package):
    """Installerer et pakke hvis den ikke allerede er installeret"""
    try:
        __import__(package)
        print(f"{package} er allerede installeret")
        return True
    except ImportError:
        print(f"Installerer {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"{package} installeret")
        return False

# Installerer spaCy
print("Tjekker og installerer spaCy...")
install_package("spacy")
print("\nSpaCy er klar!")

Tjekker og installerer spaCy...
spacy er allerede installeret

SpaCy er klar!


## Download Kinesisk Model

Efter installation af spaCy skal vi downloade en kinesisk model. Vælg en model størrelse baseret på dine behov:

- **sm** (small) - Hurtig, god til test og små projekter
- **md** (medium) - Balance mellem hastighed og kvalitet
- **lg** (large) - Bedste kvalitet, større filstørrelse
- **trf** (transformer) - Bedste performance, kræver transformers bibliotek

**Anbefaling**: Start med 'sm' for hurtig test, opgrader til 'md' eller 'lg' for bedre kvalitet.

In [2]:
# Downloader kinesisk spaCy model
# Vælg model størrelse: 'sm', 'md', 'lg', eller 'trf'
# 'sm' er anbefalet til start (hurtig og lille)

MODEL_SIZE = 'sm'  # Ændr til 'md', 'lg', eller 'trf' hvis du vil

import subprocess
import sys

model_name = f"zh_core_web_{MODEL_SIZE}"

print(f"Downloader {model_name}...")
print("Dette kan tage et par minutter første gang...")

try:
    subprocess.check_call([
        sys.executable, "-m", "spacy", "download", model_name
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(f"{model_name} downloadet og installeret!")
except subprocess.CalledProcessError as e:
    print(f"Fejl ved download: {e}")
    print("\nPrøv manuelt i terminal:")
    print(f"python -m spacy download {model_name}")

Downloader zh_core_web_sm...
Dette kan tage et par minutter første gang...
zh_core_web_sm downloadet og installeret!


## Load Model

Efter download kan vi loade den kinesiske spaCy model i hukommelsen. Modellen vil blive brugt til alle efterfølgende analyser.

In [3]:
import spacy

# Loader den kinesiske model
# Ændr 'sm' til 'md', 'lg', eller 'trf' hvis du har downloadet en anden størrelse
nlp = spacy.load("zh_core_web_sm")

print("Kinesisk spaCy model indlæst!")
print(f"Pipelinens komponenter: {nlp.pipe_names}")

Kinesisk spaCy model indlæst!
Pipelinens komponenter: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'ner']


## Eksempler på Brug

Nedenfor følger eksempler, der illustrer nogle af spaCy-bibliotekets funktioner.

### 1. Ordsegmentering (Word Segmentation)

Ordsegmentering handler om at få teksten delt op i individuelle ord. Dette vil ofte være et af de første steps i en computerbaseret tekstanalyse.

In [4]:
# Ordsegmentering
sentence = "人民是中国的主人。"
doc = nlp(sentence)

print(f"Input: {sentence}")
print("\nTokens (ord):")
for token in doc:
    print(f"  {token.text}")

Input: 人民是中国的主人。

Tokens (ord):
  人民
  是
  中国
  的
  主人
  。


### 2. Part-of-Speech Tagging (eller POS Tagging)

Part-of-Speech Tagging betyder, at hvert ord i teksten får tilføjet et tag. Et tag er det samme som et label eller et "mærke". Når taggingen er færdig, så har alle ord en henvisning til den ordklasse, som ordet tilhører (substantiv, verbum, adjektiv, osv.).

In [5]:
# POS tagging
sentence = "人民是中国的主人。"
doc = nlp(sentence)

print(f"Input: {sentence}")
print("\nOrd | Ordklasse | Beskrivelse")
print("-" * 50)
for token in doc:
    print(f"{token.text:8} | {token.pos_:8} | {spacy.explain(token.pos_)}")

Input: 人民是中国的主人。

Ord | Ordklasse | Beskrivelse
--------------------------------------------------
人民       | NOUN     | noun
是        | VERB     | verb
中国       | PROPN    | proper noun
的        | PART     | particle
主人       | NOUN     | noun
。        | PUNCT    | punctuation


### 3. Named Entity Recognition (NER)

Named Entity Recognition eller NER minder om POS-tagging. Men hvor POS tagging finder ordklasser, så finder NER så vidt muligt at finde og opmærke "navngivne enheder" ("Named Entities" på engelsk) , dvs. personnavne, geografiske stednavne og navne på organisationer.

In [6]:
# Named Entity Recognition
sentence = "北京是中国的首都。"
doc = nlp(sentence)

print(f"Input: {sentence}")
print("\nNavngivne Enheder:")
for ent in doc.ents:
    print(f"  {ent.text} - {ent.label_} ({spacy.explain(ent.label_)})")

Input: 北京是中国的首都。

Navngivne Enheder:
  北京 - GPE (Countries, cities, states)
  中国 - GPE (Countries, cities, states)


### 4. Dependency Parsing

Dependency Parsing opmærker indbyrdes, grammatiske relationer mellem ord i en sætning. Det vil sige, at programmet tilføjer et tag, der beskriver de forskellige sætningsled, hvilket giver indblik i hvilke ord, der styrer og modificerer andre ord. Det rummer nogle muligheder i forhold til at kunne analysere, hvilke ord, som teksten knytter sammen og sætter i relation til hinanden for at skabe mening. 

In [7]:
# Dependency Parsing
sentence = "人民是中国的主人。"
doc = nlp(sentence)

print(f"Input: {sentence}")
print("\nToken | Head | Dependency | Description")
print("-" * 60)
for token in doc:
    print(f"{token.text:8} | {token.head.text:8} | {token.dep_:12} | {spacy.explain(token.dep_)}")

Input: 人民是中国的主人。

Token | Head | Dependency | Description
------------------------------------------------------------
人民       | 主人       | nsubj        | nominal subject
是        | 主人       | cop          | copula
中国       | 主人       | nmod:assmod  | None
的        | 中国       | case         | case marking
主人       | 主人       | ROOT         | root
。        | 主人       | punct        | punctuation


c:\Users\lakj\AppData\Local\anaconda3\envs\Spacy\Lib\site-packages\spacy\glossary.py:20: UserWarning: [W118] Term 'nmod:assmod' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))


### 5. Visualisering af Afhængigheds-parse

spaCy kan visualisere afhængigheds-træet grafisk, hvilket gør det lettere at forstå sætningens struktur.

In [8]:
# Visualiserer afhængigheds-parse (kræver displacy)
from spacy import displacy

sentence = "人民是中国的主人。"
doc = nlp(sentence)

# Viser afhængigheds-parse
displacy.render(doc, style="dep", jupyter=True)

### 6. Batch Processing

Batch processing gør det muligt at behandle flere sætninger på én gang, hvilket er mere effektivt end at behandle dem enkeltvis.

In [9]:
# Batch processing
sentences = [
    "我爱踢足球。",
    "北京是中国的首都。",
    "今天天气很好。"
]

print("Batch processing resultater:")
print("=" * 50)

for doc in nlp.pipe(sentences):
    print(f"\nSætning: {doc.text}")
    print(f"Tokens: {[token.text for token in doc]}")
    print(f"Ordklasser: {[token.pos_ for token in doc]}")
    if doc.ents:
        print(f"Enheder: {[(ent.text, ent.label_) for ent in doc.ents]}")

Batch processing resultater:

Sætning: 我爱踢足球。
Tokens: ['我', '爱', '踢', '足球', '。']
Ordklasser: ['PRON', 'VERB', 'VERB', 'NOUN', 'PUNCT']

Sætning: 北京是中国的首都。
Tokens: ['北京', '是', '中国', '的', '首都', '。']
Ordklasser: ['PROPN', 'VERB', 'PROPN', 'PART', 'NOUN', 'PUNCT']
Enheder: [('北京', 'GPE'), ('中国', 'GPE')]

Sætning: 今天天气很好。
Tokens: ['今天', '天气', '很', '好', '。']
Ordklasser: ['NOUN', 'NOUN', 'ADV', 'VERB', 'PUNCT']
Enheder: [('今天', 'DATE')]


### 7. Detaljeret Token Information

Denne sektion viser alle tilgængelige informationer om hvert token i en sætning, inklusiv lemma, ordklasse, afhængighedsrelationer, osv.

In [10]:
# Detaljeret token information
sentence = "人民是中国的主人。"
doc = nlp(sentence)

print(f"Input: {sentence}\n")
print("Detaljeret token information:")
print("=" * 70)

for token in doc:
    print(f"\nToken: {token.text}")
    print(f"  - Lemma: {token.lemma_}")
    print(f"  - Ordklasse: {token.pos_} ({spacy.explain(token.pos_)})")
    print(f"  - Tag: {token.tag_}")
    print(f"  - Dependency: {token.dep_} ({spacy.explain(token.dep_)})")
    print(f"  - Head: {token.head.text}")
    print(f"  - Er stopord: {token.is_stop}")
    print(f"  - Er tegnsætning: {token.is_punct}")
    if token.ent_type_:
        print(f"  - Entity: {token.ent_type_} ({spacy.explain(token.ent_type_)})")

Input: 人民是中国的主人。

Detaljeret token information:

Token: 人民
  - Lemma: 
  - Ordklasse: NOUN (noun)
  - Tag: NN
  - Dependency: nsubj (nominal subject)
  - Head: 主人
  - Er stopord: True
  - Er tegnsætning: False

Token: 是
  - Lemma: 
  - Ordklasse: VERB (verb)
  - Tag: VC
  - Dependency: cop (copula)
  - Head: 主人
  - Er stopord: True
  - Er tegnsætning: False

Token: 中国
  - Lemma: 
  - Ordklasse: PROPN (proper noun)
  - Tag: NR
  - Dependency: nmod:assmod (None)
  - Head: 主人
  - Er stopord: False
  - Er tegnsætning: False
  - Entity: GPE (Countries, cities, states)

Token: 的
  - Lemma: 
  - Ordklasse: PART (particle)
  - Tag: DEG
  - Dependency: case (case marking)
  - Head: 中国
  - Er stopord: True
  - Er tegnsætning: False

Token: 主人
  - Lemma: 
  - Ordklasse: NOUN (noun)
  - Tag: NN
  - Dependency: ROOT (root)
  - Head: 主人
  - Er stopord: False
  - Er tegnsætning: False

Token: 。
  - Lemma: 
  - Ordklasse: PUNCT (punctuation)
  - Tag: PU
  - Dependency: punct (punctuation)
  - Head: 主人

## Test med dit egen tekst

Brug nedenstående celle til at teste spaCy med din egen kinesiske tekst. Den viser alle grundlæggende analyser på én gang.

In [11]:
# Test med din egen tekst
test_text = "人民是中国的主人。"

doc = nlp(test_text)

print(f"Input: {test_text}\n")
print("=" * 60)

print("\n1. Ordsegmentering:")
print([token.text for token in doc])

print("\n2. Ordklasser:")
for token in doc:
    print(f"  {token.text}: {token.pos_}")

print("\n3. Navngivne Enheder:")
if doc.ents:
    for ent in doc.ents:
        print(f"  {ent.text} ({ent.label_})")
else:
    print("  Ingen enheder fundet")

print("\n4. Dependency Parse:")
for token in doc:
    print(f"  {token.text} -> {token.head.text} ({token.dep_})")

Input: 人民是中国的主人。


1. Ordsegmentering:
['人民', '是', '中国', '的', '主人', '。']

2. Ordklasser:
  人民: NOUN
  是: VERB
  中国: PROPN
  的: PART
  主人: NOUN
  。: PUNCT

3. Navngivne Enheder:
  中国 (GPE)

4. Dependency Parse:
  人民 -> 主人 (nsubj)
  是 -> 主人 (cop)
  中国 -> 主人 (nmod:assmod)
  的 -> 中国 (case)
  主人 -> 主人 (ROOT)
  。 -> 主人 (punct)


## Find Modifiers for et Navneord

Denne sektion indeholder funktioner til at finde og analysere ord der modificerer navneord i kinesisk tekst. Dette er nyttigt til at forstå hvordan navneord bliver beskrevet eller kvalificeret i sætninger.

Funktionen bruger afhængigheds-parsing til at identificere grammatiske relationer mellem ord.

In [12]:
def find_noun_modifiers(doc, target_noun):
    """
    Finder alle ord der modificerer et givet navneord.
    
    Parametre:
        doc: spaCy Doc objekt
        target_noun: Navneordet du vil finde modifiers for (som string)
    
    Returnerer:
        Tuple: (modifiers_list, target_token) eller (None, None) hvis ikke fundet
    """
    modifiers = []
    
    # Finder navneordet i dokumentet
    target_token = None
    for token in doc:
        if token.text == target_noun or token.lemma_ == target_noun:
            target_token = token
            break
    
    if target_token is None:
        return None, None
    
    # Finder alle tokens der har target_token som hoved (dvs. modificerer det)
    for token in doc:
        if token.head == target_token and token != target_token:
            # Forskellige typer af modifiers
            if token.dep_ in ['amod', 'nmod', 'nummod', 'det', 'poss', 'case']:
                modifiers.append({
                    'text': token.text,
                    'dependency': token.dep_,
                    'pos': token.pos_,
                    'description': spacy.explain(token.dep_)
                })
            # For kinesisk: også inkluder nmod:assmod (associativ modifier)
            elif 'nmod' in token.dep_ or 'amod' in token.dep_:
                modifiers.append({
                    'text': token.text,
                    'dependency': token.dep_,
                    'pos': token.pos_,
                    'description': spacy.explain(token.dep_)
                })
    
    # Finder også tokens der modificerer via compound eller andre relationer
    for token in doc:
        if token.head == target_token:
            # Inkluder også compound modifiers
            if 'compound' in token.dep_ or 'nmod' in token.dep_:
                # Undgår duplikater
                if not any(m['text'] == token.text for m in modifiers):
                    modifiers.append({
                        'text': token.text,
                        'dependency': token.dep_,
                        'pos': token.pos_,
                        'description': spacy.explain(token.dep_)
                    })
    
    return modifiers, target_token

# Eksempel brug
sentence = "美丽的花朵在花园里绽放。"
target_noun = "花朵"  # Navneordet du vil finde modifiers for

doc = nlp(sentence)
modifiers, noun_token = find_noun_modifiers(doc, target_noun)

print(f"Sætning: {sentence}")
print(f"\nNavneord: {target_noun}")
print(f"Ordklasse: {noun_token.pos_} ({spacy.explain(noun_token.pos_)})")
print("\n" + "=" * 60)
print("Modifiers:")
print("=" * 60)

if modifiers is None or noun_token is None:
    print(f"Navneordet '{target_noun}' blev ikke fundet i teksten.")
else:
    if modifiers:
        for mod in modifiers:
            print(f"\n  Modifier: {mod['text']}")
            print(f"    - Afhængighed: {mod['dependency']} ({mod['description']})")
            print(f"    - Ordklasse: {mod['pos']} ({spacy.explain(mod['pos'])})")
    else:
        print("  Ingen modifiers fundet for dette navneord.")

Sætning: 美丽的花朵在花园里绽放。

Navneord: 花朵
Ordklasse: NOUN (noun)

Modifiers:

  Modifier: 美丽
    - Afhængighed: amod (adjectival modifier)
    - Ordklasse: VERB (verb)


In [13]:
# Interaktiv funktion: Indtast din tekst og navneord
def get_noun_modifiers_interactive(text, noun):
    """
    Interaktiv funktion til at finde modifiers for et navneord.
    
    Parametre:
        text: Kinesisk tekst
        noun: Navneordet du vil analysere
    """
    doc = nlp(text)
    modifiers, noun_token = find_noun_modifiers(doc, noun)
    
    print("=" * 70)
    print(f"TEKST: {text}")
    print("=" * 70)
    print(f"\nNAVNEORD: {noun}")
    
    if noun_token is None:
        print(f"\n Navneordet '{noun}' blev ikke fundet i teksten.")
        print("\nTilgængelige navneord i teksten:")
        nouns = [token.text for token in doc if token.pos_ in ['NOUN', 'PROPN']]
        if nouns:
            print(f"  {', '.join(nouns)}")
        else:
            print("  Ingen navneord fundet i teksten.")
        return None
    
    print(f"Ordklasse: {noun_token.pos_} ({spacy.explain(noun_token.pos_)})")
    print(f"\nANTAL MODIFIERS: {len(modifiers)}")
    
    if modifiers:
        print("\n" + "-" * 70)
        print("MODIFIERS:")
        print("-" * 70)
        for i, mod in enumerate(modifiers, 1):
            print(f"\n{i}. {mod['text']}")
            print(f"   Dependency: {mod['dependency']} - {mod['description']}")
            print(f"   Ordklasse: {mod['pos']} ({spacy.explain(mod['pos'])})")
        
        # Viser også afhængigheds-parse for kontekst
        print("\n" + "-" * 70)
        print("DEPENDENCY PARSE (for kontekst):")
        print("-" * 70)
        for token in doc:
            if token.head == noun_token or token == noun_token:
                arrow = "→" if token.head == noun_token else "←"
                print(f"  {token.text:8} {arrow} {noun_token.text:8} ({token.dep_})")
    else:
        print("\nIngen modifiers fundet.")
        print("\nDette kan betyde:")
        print("- Navneordet er ikke modificeret i denne sætning")
        print("- Navneordet står alene uden modifiers")
    
    return modifiers

# Eksempel 1
print("EKSEMPEL 1:")
print("=" * 70)
get_noun_modifiers_interactive("美丽的花朵在花园里绽放。", "花朵")

print("\n\n")

# Eksempel 2
print("EKSEMPEL 2:")
print("=" * 70)
get_noun_modifiers_interactive("人民是中国的主人。", "主人")

EKSEMPEL 1:
TEKST: 美丽的花朵在花园里绽放。

NAVNEORD: 花朵
Ordklasse: NOUN (noun)

ANTAL MODIFIERS: 1

----------------------------------------------------------------------
MODIFIERS:
----------------------------------------------------------------------

1. 美丽
   Dependency: amod - adjectival modifier
   Ordklasse: VERB (verb)

----------------------------------------------------------------------
DEPENDENCY PARSE (for kontekst):
----------------------------------------------------------------------
  美丽       → 花朵       (amod)
  花朵       ← 花朵       (nsubj)



EKSEMPEL 2:
TEKST: 人民是中国的主人。

NAVNEORD: 主人
Ordklasse: NOUN (noun)

ANTAL MODIFIERS: 1

----------------------------------------------------------------------
MODIFIERS:
----------------------------------------------------------------------

1. 中国
   Dependency: nmod:assmod - None
   Ordklasse: PROPN (proper noun)

----------------------------------------------------------------------
DEPENDENCY PARSE (for kontekst):
------------------------

[{'text': '中国',
  'dependency': 'nmod:assmod',
  'pos': 'PROPN',
  'description': None}]

### Brug med din egen tekst

Brug nedenstående celle til at indtaste din egen tekst og navneord. Funktionen vil automatisk finde alle ord der modificerer navneordet.

In [14]:
# Indtast din egen tekst og navneord her
din_tekst = "国家旅游局发言人发表谈话  来华游览活动可正常进行  国旅总社总经理说旅游者人身安全未受侵犯"  # Ændr til din tekst
dit_navneord = "发言人"  # Ændr til navneordet du vil analysere

# Kører analysen
resultat = get_noun_modifiers_interactive(din_tekst, dit_navneord)

TEKST: 国家旅游局发言人发表谈话  来华游览活动可正常进行  国旅总社总经理说旅游者人身安全未受侵犯

NAVNEORD: 发言人
Ordklasse: NOUN (noun)

ANTAL MODIFIERS: 2

----------------------------------------------------------------------
MODIFIERS:
----------------------------------------------------------------------

1. 国家
   Dependency: compound:nn - None
   Ordklasse: NOUN (noun)

2. 旅游局
   Dependency: compound:nn - None
   Ordklasse: NOUN (noun)

----------------------------------------------------------------------
DEPENDENCY PARSE (for kontekst):
----------------------------------------------------------------------
  国家       → 发言人      (compound:nn)
  旅游局      → 发言人      (compound:nn)
  发言人      ← 发言人      (nsubj)


c:\Users\lakj\AppData\Local\anaconda3\envs\Spacy\Lib\site-packages\spacy\glossary.py:20: UserWarning: [W118] Term 'compound:nn' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))


### Find alle Navneord og deres Modifiers

Denne funktion finder automatisk alle navneord i en tekst og viser deres modifiers. Dette er nyttigt til at analysere hele dokumenter eller længere tekster.

In [15]:
def find_all_nouns_with_modifiers(text):
    """
    Finder alle navneord i en tekst og deres modifiers.
    """
    doc = nlp(text)
    
    # Finder alle navneord (NOUN, PROPN)
    nouns = [token for token in doc if token.pos_ in ['NOUN', 'PROPN']]
    
    print(f"TEKST: {text}")
    print("=" * 70)
    print(f"\nFUNDET {len(nouns)} NAVNEORD:\n")
    
    for i, noun in enumerate(nouns, 1):
        print(f"{i}. NAVNEORD: {noun.text}")
        print(f"   Ordklasse: {noun.pos_} ({spacy.explain(noun.pos_)})")
        
        # Finder modifiers for dette navneord
        modifiers = []
        for token in doc:
            if token.head == noun and token != noun:
                if token.dep_ in ['amod', 'nmod', 'nummod', 'det', 'poss', 'case'] or \
                   'nmod' in token.dep_ or 'amod' in token.dep_ or 'compound' in token.dep_:
                    modifiers.append({
                        'text': token.text,
                        'dep': token.dep_,
                        'pos': token.pos_
                    })
        
        if modifiers:
            print(f"   MODIFIERS: {', '.join([m['text'] for m in modifiers])}")
            for mod in modifiers:
                print(f"      - {mod['text']} ({mod['dep']}, {mod['pos']})")
        else:
            print(f"   MODIFIERS: Ingen")
        print()
    
    return nouns

# Eksempel
find_all_nouns_with_modifiers("据新华社北京６月２日电　（记者陈芸）中国国际旅行社集团董事长、中国国际旅行社总社总经理王尔康今天在此间对记者说，从４月１５日至今，来华旅游者的人身安全没有受到任何侵犯，也没有受到不礼貌或不愉快的遭遇，他们的旅游计划基本正常进行。")

TEKST: 据新华社北京６月２日电　（记者陈芸）中国国际旅行社集团董事长、中国国际旅行社总社总经理王尔康今天在此间对记者说，从４月１５日至今，来华旅游者的人身安全没有受到任何侵犯，也没有受到不礼貌或不愉快的遭遇，他们的旅游计划基本正常进行。

FUNDET 30 NAVNEORD:

1. NAVNEORD: 新华社
   Ordklasse: PROPN (proper noun)
   MODIFIERS: Ingen

2. NAVNEORD: 北京
   Ordklasse: PROPN (proper noun)
   MODIFIERS: Ingen

3. NAVNEORD: ６月
   Ordklasse: NOUN (noun)
   MODIFIERS: Ingen

4. NAVNEORD: ２日
   Ordklasse: NOUN (noun)
   MODIFIERS: Ingen

5. NAVNEORD: 电
   Ordklasse: NOUN (noun)
   MODIFIERS: Ingen

6. NAVNEORD: 记者
   Ordklasse: NOUN (noun)
   MODIFIERS: Ingen

7. NAVNEORD: 陈芸
   Ordklasse: PROPN (proper noun)
   MODIFIERS: Ingen

8. NAVNEORD: 中国
   Ordklasse: PROPN (proper noun)
   MODIFIERS: Ingen

9. NAVNEORD: 国际
   Ordklasse: NOUN (noun)
   MODIFIERS: Ingen

10. NAVNEORD: 旅行社
   Ordklasse: NOUN (noun)
   MODIFIERS: Ingen

11. NAVNEORD: 集团
   Ordklasse: NOUN (noun)
   MODIFIERS: 中国, 国际, 旅行社
      - 中国 (compound:nn, PROPN)
      - 国际 (compound:nn, NOUN)
      - 旅行社 (compound:nn, NOUN)

12. NAVNEORD: 董事长
   Ordkla

[新华社,
 北京,
 ６月,
 ２日,
 电,
 记者,
 陈芸,
 中国,
 国际,
 旅行社,
 集团,
 董事长,
 中国,
 国际,
 旅行社,
 总社,
 总经理,
 王尔康,
 今天,
 此间,
 记者,
 ４月,
 １５日,
 旅游者,
 人身,
 安全,
 侵犯,
 遭遇,
 旅游,
 计划]

## Yderligere Ressourcer

For mere information om spaCy's kinesiske modeller og funktioner, se:

- [spaCy Kinesiske Modeller Dokumentation](https://spacy.io/models/zh)
- [spaCy Brugsguide](https://spacy.io/usage/models#languages)
- [spaCy API Reference](https://spacy.io/api/doc)